# HoloSyn: Multimodal P2P Co‑Regulation — Gemini Teacher Distillation (Colab)

This notebook is **from scratch** and intended to be run in **Google Colab**.

It builds a real, end‑to‑end *multimodal distillation pipeline*:

- Extract **audio / video / images / text / haptics** from `Archive.zip`
- Create **teacher labels** using **Gemini** (requires `GEMINI_API_KEY`)
- Train a **small student model** (PyTorch) to predict the teacher labels
- Optionally add **Cirq/qsimcirq** (quantum feature map) + **Brian2** (entrainment/stress dynamics)
- Export a deployable **TorchScript** student head

> Privacy-first default: send **small clips / sampled frames / transcripts / summaries** to Gemini, not entire raw archives.

---

## Inputs expected (already uploaded to this runtime)
- `/mnt/data/Archive.zip`
- `/mnt/data/holosyn_heads.pt` (optional; used if you want to initialize/extend)
- `/mnt/data/holosyn_heads.torchscript.pt` (optional; deployment artifact)

If you're running in Colab, you can also upload these to `/content/`.


In [ ]:
#@title 0) Install dependencies
!pip -q install -U google-genai
!pip -q install -U numpy pandas tqdm pillow opencv-python soundfile librosa
!pip -q install -U torch torchvision torchaudio
!pip -q install -U cirq qsimcirq brian2
!pip -q install -U cryptography

import os, json, math, zipfile, pathlib, time, hashlib, random
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image

print("✅ Installed.")

## 1) Set paths

Colab usually uses `/content`. This notebook also supports the sandbox path `/mnt/data`.


In [ ]:
#@title 1) Locate files
from pathlib import Path

def pick_existing(*candidates):
    for c in candidates:
        if c and os.path.exists(c):
            return c
    return None

ARCHIVE_ZIP = pick_existing("/mnt/data/Archive.zip", "/content/Archive.zip")
HEADS_PT    = pick_existing("/mnt/data/holosyn_heads.pt", "/content/holosyn_heads.pt")
HEADS_TS    = pick_existing("/mnt/data/holosyn_heads.torchscript.pt", "/content/holosyn_heads.torchscript.pt")

assert ARCHIVE_ZIP, "❌ Archive.zip not found. Upload it to Colab (Files pane) or place in /mnt/data."

EXTRACT_DIR = "/content/archive_extracted"
Path(EXTRACT_DIR).mkdir(parents=True, exist_ok=True)

print("ARCHIVE_ZIP:", ARCHIVE_ZIP)
print("HEADS_PT   :", HEADS_PT)
print("HEADS_TS   :", HEADS_TS)
print("EXTRACT_DIR:", EXTRACT_DIR)

In [ ]:
#@title 2) Extract Archive.zip
import zipfile, pathlib

with zipfile.ZipFile(ARCHIVE_ZIP, "r") as z:
    z.extractall(EXTRACT_DIR)

# show top-level structure
top = sorted([p for p in pathlib.Path(EXTRACT_DIR).iterdir()])
print("Top-level items:")
for p in top[:50]:
    print(" -", p.name, "(dir)" if p.is_dir() else "(file)")

## 3) Index files by modality

In [ ]:
#@title 3) Discover files (audio/video/image/text/haptics)
import mimetypes

AUDIO_EXT = {".wav",".mp3",".m4a",".flac",".ogg",".aac"}
VIDEO_EXT = {".mp4",".mov",".mkv",".webm",".avi"}
IMAGE_EXT = {".png",".jpg",".jpeg",".webp",".bmp"}
TEXT_EXT  = {".txt",".md",".json",".csv",".tsv"}
HAPTIC_EXT = {".hapt",".haptic",".pattern",".vib",".json",".csv",".tsv"}  # adjust if needed

def walk_files(root):
    out = []
    for p in Path(root).rglob("*"):
        if p.is_file():
            out.append(str(p))
    return out

def bucket(path):
    ext = Path(path).suffix.lower()
    if ext in AUDIO_EXT: return "audio"
    if ext in VIDEO_EXT: return "video"
    if ext in IMAGE_EXT: return "image"
    # disambiguate JSON: treat as haptics if path hints it; else text
    if ext == ".json":
        low = path.lower()
        if any(k in low for k in ["hapt", "vib", "pattern", "tact", "rumble"]):
            return "haptics"
        return "text"
    if ext in TEXT_EXT:  return "text"
    if ext in HAPTIC_EXT and any(k in path.lower() for k in ["hapt", "vib", "pattern", "tact", "rumble"]):
        return "haptics"
    return "other"

all_files = walk_files(EXTRACT_DIR)
buckets = {}
for f in all_files:
    buckets.setdefault(bucket(f), []).append(f)

for k in ["audio","video","image","text","haptics","other"]:
    print(f"{k:8s}", len(buckets.get(k, [])))
    samp = buckets.get(k, [])[:3]
    for s in samp:
        print("   •", s.replace(EXTRACT_DIR + "/", ""))

## 4) Feature extraction (local)

We extract **compact, privacy-preserving features** locally.  
These will be used as student inputs; Gemini provides teacher labels.

### Outputs:
- `features.parquet` (tabular features)
- `teacher_labels.jsonl` (Gemini labels cache)


In [ ]:
#@title 4A) Helpers: load text, haptics, images, audio features, video frames
import soundfile as sf
import librosa
import cv2

def load_text(path, max_chars=12000):
    s = open(path, "r", encoding="utf-8", errors="ignore").read()
    return s[:max_chars]

def load_haptics_any(path):
    ext = Path(path).suffix.lower()
    if ext == ".json":
        try:
            return json.load(open(path, "r", encoding="utf-8", errors="ignore"))
        except Exception:
            return {"raw": load_text(path, 12000)}
    if ext in {".csv",".tsv"}:
        sep = "," if ext==".csv" else "\t"
        try:
            df = pd.read_csv(path, sep=sep)
            return {"columns": list(df.columns), "head": df.head(200).to_dict(orient="list")}
        except Exception:
            return {"raw": load_text(path, 12000)}
    return {"raw": load_text(path, 12000)}

def featurize_text(s):
    # simple, robust lexical stats
    return {
        "txt_len": float(len(s)),
        "txt_lines": float(s.count("\n")+1),
        "txt_exclaim": float(s.count("!")),
        "txt_question": float(s.count("?")),
        "txt_caps_ratio": float(sum(c.isupper() for c in s)/max(1,len(s))),
    }

def featurize_haptics(h):
    raw = json.dumps(h)[:20000]
    return {
        "hapt_len": float(len(raw)),
        "hapt_has_intensity": 1.0 if "intensity" in raw.lower() else 0.0,
        "hapt_has_freq": 1.0 if "hz" in raw.lower() or "freq" in raw.lower() else 0.0,
    }

def audio_features(path, sr=16000, max_seconds=15):
    # returns compact prosody-ish features
    y, file_sr = librosa.load(path, sr=sr, mono=True, duration=max_seconds)
    if y.size < 10:
        return {"aud_rms":0.0,"aud_zcr":0.0,"aud_centroid":0.0,"aud_tempo":0.0}
    rms = float(np.mean(librosa.feature.rms(y=y)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    tempo = float(librosa.beat.tempo(onset_envelope=onset_env, sr=sr)[0]) if onset_env.size else 0.0
    return {"aud_rms":rms,"aud_zcr":zcr,"aud_centroid":centroid,"aud_tempo":tempo}

def sample_video_frames(video_path, every_n_frames=30, max_frames=6, target_size=224):
    cap = cv2.VideoCapture(video_path)
    frames = []
    idx, got = 0, 0
    while cap.isOpened() and got < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % every_n_frames == 0:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            # resize
            h, w = frame.shape[:2]
            if max(h,w) > target_size:
                scale = target_size / max(h,w)
                frame = cv2.resize(frame, (int(w*scale), int(h*scale)))
            frames.append(frame)
            got += 1
        idx += 1
    cap.release()
    return frames

def image_quick_stats(path):
    im = Image.open(path).convert("RGB")
    arr = np.asarray(im).astype(np.float32) / 255.0
    # simple stats
    mean = arr.mean(axis=(0,1))
    std = arr.std(axis=(0,1))
    return {
        "img_w": float(arr.shape[1]),
        "img_h": float(arr.shape[0]),
        "img_mean_r": float(mean[0]),
        "img_mean_g": float(mean[1]),
        "img_mean_b": float(mean[2]),
        "img_std_r": float(std[0]),
        "img_std_g": float(std[1]),
        "img_std_b": float(std[2]),
    }

In [ ]:
#@title 4B) Build feature table
MAX_PER_MODALITY = 300   # adjust
rows = []

def rel(p): return p.replace(EXTRACT_DIR + "/", "")

# TEXT
for p in tqdm(buckets.get("text", [])[:MAX_PER_MODALITY], desc="text"):
    s = load_text(p)
    feats = featurize_text(s)
    rows.append({"path": p, "rel": rel(p), "modality":"text", **feats})

# HAPTICS
for p in tqdm(buckets.get("haptics", [])[:MAX_PER_MODALITY], desc="haptics"):
    h = load_haptics_any(p)
    feats = featurize_haptics(h)
    rows.append({"path": p, "rel": rel(p), "modality":"haptics", **feats})

# AUDIO
for p in tqdm(buckets.get("audio", [])[:MAX_PER_MODALITY], desc="audio"):
    feats = audio_features(p)
    rows.append({"path": p, "rel": rel(p), "modality":"audio", **feats})

# IMAGE
for p in tqdm(buckets.get("image", [])[:MAX_PER_MODALITY], desc="image"):
    try:
        feats = image_quick_stats(p)
        rows.append({"path": p, "rel": rel(p), "modality":"image", **feats})
    except Exception as e:
        print("skip image", rel(p), e)

# VIDEO (stats only; frames used later for teacher)
for p in tqdm(buckets.get("video", [])[:MAX_PER_MODALITY], desc="video"):
    try:
        frames = sample_video_frames(p, every_n_frames=45, max_frames=4, target_size=224)
        rows.append({"path": p, "rel": rel(p), "modality":"video", "vid_n_frames": float(len(frames))})
    except Exception as e:
        print("skip video", rel(p), e)

df = pd.DataFrame(rows).fillna(0.0)
df_path = "/content/features.parquet"
df.to_parquet(df_path, index=False)
print("✅ Features:", df.shape, "saved to", df_path)
df.head()

## 5) Gemini teacher labeling (requires `GEMINI_API_KEY`)

### Set the key (DO NOT paste into chat)
In Colab:
- Left sidebar → **Secrets** → add `GEMINI_API_KEY`
OR run:
```python
import os
os.environ["GEMINI_API_KEY"] = "..."
```

We cache labels to JSONL so you can resume without re-paying for labels.


In [ ]:
#@title 5A) Gemini client + label schema
import os, json
from google import genai

assert os.environ.get("GEMINI_API_KEY"), "❌ Set GEMINI_API_KEY in Colab Secrets or os.environ."

client = genai.Client()
TEACHER_MODEL = "gemini-2.5-flash"

LABEL_KEYS = ["valence","arousal","calm","trust","pacing_suggestion","summary"]

def parse_json_strict(s: str) -> dict:
    # tolerate code fences but require JSON object
    s = s.strip()
    if s.startswith("```"):
        s = s.split("```", 2)[1]
    s = s.strip()
    obj = json.loads(s)
    # ensure keys
    out = {k: obj.get(k, None) for k in LABEL_KEYS}
    return out

def teacher_prompt(modality: str, payload_desc: str) -> str:
    return f'''
You are labeling data for a consent-first co-regulation / intimacy interface (NON-EXPLICIT).
Return ONLY valid JSON with keys:
- valence (0..1)
- arousal (0..1)
- calm (0..1)
- trust (0..1)
- pacing_suggestion (short string)
- summary (short string)

MODALITY: {modality}
PAYLOAD:
{payload_desc}
'''.strip()

print("✅ Gemini ready with", TEACHER_MODEL)

In [ ]:
#@title 5B) Teacher label functions (text / haptics / image / video-frames / audio-summary)
from PIL import Image

def label_text(text: str) -> dict:
    prompt = teacher_prompt("text", text[:12000])
    resp = client.models.generate_content(model=TEACHER_MODEL, contents=prompt)
    return parse_json_strict(resp.text)

def label_haptics(h: dict) -> dict:
    payload = "HAPTICS_JSON:\n" + json.dumps(h)[:12000]
    prompt = teacher_prompt("haptics", payload)
    resp = client.models.generate_content(model=TEACHER_MODEL, contents=prompt)
    return parse_json_strict(resp.text)

def label_image(path: str) -> dict:
    im = Image.open(path).convert("RGB")
    prompt = teacher_prompt("image", "See attached image.")
    resp = client.models.generate_content(model=TEACHER_MODEL, contents=[prompt, im])
    return parse_json_strict(resp.text)

def label_video(path: str) -> dict:
    frames = sample_video_frames(path, every_n_frames=45, max_frames=6, target_size=224)
    pil_frames = [Image.fromarray(f).convert("RGB") for f in frames]
    prompt = teacher_prompt("video", "Sampled frames attached in order.")
    resp = client.models.generate_content(model=TEACHER_MODEL, contents=[prompt, *pil_frames])
    return parse_json_strict(resp.text)

def label_audio(path: str) -> dict:
    # privacy-first default: send a LOCAL summary of audio features (not raw audio)
    feats = audio_features(path)
    prompt = teacher_prompt("audio_features", json.dumps({"audio_features": feats}))
    resp = client.models.generate_content(model=TEACHER_MODEL, contents=prompt)
    return parse_json_strict(resp.text)

In [ ]:
#@title 5C) Run teacher labeling with caching (resume-safe)
LABEL_CACHE = "/content/teacher_labels.jsonl"
DONE = set()

if os.path.exists(LABEL_CACHE):
    with open(LABEL_CACHE, "r", encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
                DONE.add(rec["rel"])
            except:
                pass
print("Cached labels:", len(DONE))

def safe_sleep(i):
    # simple rate limiting/backoff
    time.sleep(0.6 + 0.2*(i % 3))

def label_one(row):
    mod = row["modality"]
    p = row["path"]
    if mod == "text":
        return label_text(load_text(p))
    if mod == "haptics":
        return label_haptics(load_haptics_any(p))
    if mod == "image":
        return label_image(p)
    if mod == "video":
        return label_video(p)
    if mod == "audio":
        return label_audio(p)
    return None

# choose how many to label now
N_LABEL = 200
to_label = df[~df["rel"].isin(DONE)].head(N_LABEL)

new_count = 0
with open(LABEL_CACHE, "a", encoding="utf-8") as f:
    for i, row in tqdm(list(to_label.iterrows()), total=len(to_label), desc="Gemini labeling"):
        relp = row["rel"]
        try:
            y = label_one(row)
            if y is None:
                continue
            rec = {"rel": relp, "modality": row["modality"], **y}
            f.write(json.dumps(rec) + "\n")
            new_count += 1
        except Exception as e:
            print("❌ label fail:", relp, "err:", e)
            # backoff
            time.sleep(2.0)
        safe_sleep(i)

print("✅ New labels added:", new_count)

## 6) Train student model (distillation)

We train a compact student to predict `[valence, arousal, calm, trust]` from local features.

You can expand the feature set later (embeddings, CLIP, Wav2Vec, etc.).


In [ ]:
#@title 6A) Load cached labels and merge with features
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

labels = []
if os.path.exists(LABEL_CACHE):
    with open(LABEL_CACHE, "r", encoding="utf-8") as f:
        for line in f:
            try:
                labels.append(json.loads(line))
            except:
                pass

lab = pd.DataFrame(labels)
print("Labels:", lab.shape)
merged = df.merge(lab, on=["rel","modality"], how="inner")
merged = merged.dropna(subset=["valence","arousal","calm","trust"])
print("Merged:", merged.shape)
merged.head()

In [ ]:
#@title 6B) Build feature matrix (auto-select numeric cols)
TARGETS = ["valence","arousal","calm","trust"]

numeric_cols = []
for c in merged.columns:
    if c in ["path","rel","modality","pacing_suggestion","summary"] + TARGETS:
        continue
    if pd.api.types.is_numeric_dtype(merged[c]):
        numeric_cols.append(c)

print("Numeric features:", len(numeric_cols))
print(numeric_cols[:30])

X = merged[numeric_cols].astype(np.float32).values
Y = merged[TARGETS].astype(np.float32).values

# normalize
mu = X.mean(axis=0, keepdims=True)
sd = X.std(axis=0, keepdims=True) + 1e-6
Xn = (X - mu) / sd

print("Xn", Xn.shape, "Y", Y.shape)

In [ ]:
#@title 6C) Train student MLP
class TableDS(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i], self.Y[i]

ds = TableDS(Xn, Y)
dl = DataLoader(ds, batch_size=64, shuffle=True)

student = nn.Sequential(
    nn.Linear(Xn.shape[1], 128),
    nn.Tanh(),
    nn.Linear(128, 64),
    nn.Tanh(),
    nn.Linear(64, 4),
    nn.Sigmoid(),
)

opt = torch.optim.Adam(student.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

student.train()
for epoch in range(15):
    total = 0.0
    for xb, yb in dl:
        pred = student(xb)
        loss = loss_fn(pred, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item() * xb.size(0)
    print(f"epoch {epoch:02d} mse {total/len(ds):.6f}")

student.eval()


In [ ]:
#@title 6D) Export student + normalization stats
student_ts_path = "/content/student_distilled_heads.torchscript.pt"
example = torch.zeros(1, Xn.shape[1])
ts = torch.jit.trace(student, example)
ts.save(student_ts_path)

norm_path = "/content/student_norm.json"
with open(norm_path, "w", encoding="utf-8") as f:
    json.dump({"numeric_cols": numeric_cols, "mu": mu.flatten().tolist(), "sd": sd.flatten().tolist()}, f)

print("✅ Saved TorchScript:", student_ts_path)
print("✅ Saved norm stats :", norm_path)

## 7) Optional: Quantum + Brian2 “synchrony & stress” module

This is a *runtime* module used after distillation:
- Student predicts state for each peer
- Cirq/qsimcirq produces a nonlinear synchrony feature
- Brian2 simulates entrainment and outputs a stress/overload scalar


In [ ]:
#@title 7A) Quantum synchrony (Cirq + qsimcirq)
import cirq, qsimcirq

class QuantumSynchrony:
    def __init__(self, n_qubits=4, depth=2):
        self.n_qubits = n_qubits
        self.depth = depth
        self.qubits = cirq.LineQubit.range(n_qubits)
        self.sim = qsimcirq.QSimSimulator()

    def _reduce(self, e):
        x = np.tanh(np.asarray(e, dtype=np.float32)) * np.pi
        x = x[:self.n_qubits]
        if x.size < self.n_qubits:
            x = np.pad(x, (0, self.n_qubits-x.size))
        return x

    def _circuit(self, x):
        c = cirq.Circuit()
        c.append([cirq.H(q) for q in self.qubits])
        for _ in range(self.depth):
            for i,q in enumerate(self.qubits):
                c.append(cirq.ry(x[i])(q))
                c.append(cirq.rz(0.5*x[i])(q))
            for i in range(self.n_qubits-1):
                c.append(cirq.CZ(self.qubits[i], self.qubits[i+1]))
        return c

    def kernel(self, ea, eb):
        xa = self._reduce(ea)
        xb = self._reduce(eb)
        sa = self.sim.simulate(self._circuit(xa)).final_state_vector
        sb = self.sim.simulate(self._circuit(xb)).final_state_vector
        fid = float(np.abs(np.vdot(sa, sb))**2)
        return fid

qs = QuantumSynchrony(n_qubits=4, depth=2)
print("qsync demo:", qs.kernel([0.1,0.2,0.3,0.4],[0.1,0.2,0.3,0.4]))

In [ ]:
#@title 7B) Brian2 entrainment (fast coupled oscillators)
from brian2 import *

def plv_from_coupling(coupling=0.2, duration_ms=200):
    start_scope()
    eqs = '''
    dtheta/dt = omega + k*sin(theta_other - theta) : 1
    theta_other : 1
    omega : 1
    k : 1
    '''
    G = NeuronGroup(2, eqs, method='euler')
    G.theta = [0.1, 2.0]
    G.omega = [2*np.pi*8, 2*np.pi*8]
    G.k = coupling

    @network_operation(dt=1*ms)
    def couple():
        G.theta_other[0] = G.theta[1]
        G.theta_other[1] = G.theta[0]

    M = StateMonitor(G, 'theta', record=True)
    net = Network(G, couple, M)
    net.run(duration_ms*ms)

    phase_diff = np.array(M.theta[0] - M.theta[1])
    plv = float(np.abs(np.mean(np.exp(1j*phase_diff))))
    return plv

print("plv demo:", plv_from_coupling(0.05), plv_from_coupling(0.35))

## 8) Observer Cookie (local, signed, updateable)

This is the updatable observer state you keep locally (and optionally exchange with the peer).
In production: encrypt and store in IndexedDB / keystore.


In [ ]:
#@title 8) Signed observer cookie
import hmac, base64
from dataclasses import dataclass, asdict
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import secrets

def b64e(b: bytes) -> str:
    return base64.urlsafe_b64encode(b).decode("utf-8").rstrip("=")

def b64d(s: str) -> bytes:
    pad = "=" * (-len(s) % 4)
    return base64.urlsafe_b64decode(s + pad)

def hmac_sign(key: bytes, msg: bytes) -> str:
    return b64e(hmac.new(key, msg, hashlib.sha256).digest())

def hmac_verify(key: bytes, msg: bytes, sig: str) -> bool:
    return hmac.compare_digest(b64d(sig), b64d(hmac_sign(key, msg)))

@dataclass
class ObserverCookie:
    version: int
    session_id: str
    created_ts: float
    updated_ts: float
    counter: int
    metrics: dict
    boundaries: dict

    def to_bytes(self) -> bytes:
        return json.dumps(asdict(self), sort_keys=True, separators=(",",":")).encode("utf-8")

class CookieSigner:
    def __init__(self, key=None):
        self.key = key or secrets.token_bytes(32)
    def pack(self, c: ObserverCookie) -> dict:
        payload = c.to_bytes()
        return {"payload": b64e(payload), "sig": hmac_sign(self.key, payload)}
    def unpack(self, blob: dict) -> ObserverCookie:
        payload = b64d(blob["payload"])
        if not hmac_verify(self.key, payload, blob["sig"]):
            raise ValueError("Cookie tampered")
        return ObserverCookie(**json.loads(payload.decode("utf-8")))

def new_cookie():
    sid = b64e(secrets.token_bytes(12))
    now = time.time()
    return ObserverCookie(
        version=1, session_id=sid, created_ts=now, updated_ts=now, counter=0,
        metrics={"safety":0.7,"attunement":0.5,"pace":0.5,"trust":0.5},
        boundaries={"max_intensity":0.6,"consent_required":True}
    )

signer = CookieSigner()
cookie = new_cookie()
blob = signer.pack(cookie)
cookie2 = signer.unpack(blob)
print("cookie session:", cookie2.session_id, "ok")

## 9) Next: wire into real P2P runtime

This notebook builds the *modeling + distillation* side.  
To make it fully P2P:
- Use WebRTC datachannels (browser or mobile)
- Send: **quantized student outputs**, **consent state**, and **haptic intents**
- Keep raw A/V local unless both opt-in.

If you want, say what platform you’re targeting (web / iOS / Android / desktop), and I’ll generate the P2P runtime notebook too.
